# Week 4 — Prompt Engineering: Extraction, Math, and Chain of Thought

This notebook covers two problems:

1. **Problem #1** — Extract structured order data (plus prices, subtotals, and a grand total) from customer emails as JSON, first with a **zero-shot** prompt and then with a **one-shot** prompt, using OpenAI.
2. **Problem #2** — Re-create the chain-of-thought chat from Listing 6.4 using the **OpenAI** API (not Azure OpenAI).

> **Before submitting:** the sections marked **✏️ Your commentary** need to be written after you run the notebook, because they depend on what the model actually returned on your run. Guiding questions are included in each one; replace them with your own observations.

## Setup

Both problems use the OpenAI API and the OpenAI Python SDK. Problem 1 compares zero-shot and one-shot prompting with the same model; Problem 2 uses the same OpenAI model for the chain-of-thought exercise.

You need an OpenAI API key stored as `OPENAI_API_KEY`. If it is not set in the environment, the setup cell prompts for it without echoing it. API usage may incur charges.

In [1]:
%pip install -q --upgrade openai

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import json
import re
from getpass import getpass
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Set it in the environment or enter it when prompted."
    )

oa_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
OPENAI_MODEL = "gpt-4o-mini"
print(f"OpenAI client initialized with {OPENAI_MODEL}.")

OpenAI client initialized with gpt-4o-mini.


---
# Problem #1 — Extraction and Math with Zero-Shot and One-Shot Prompting

## Approach

The task mixes two very different skills: **extraction** (reading names, addresses, and line items out of free text), which LLMs are good at, and **arithmetic** (quantity × price, then a sum), which LLMs are notoriously unreliable at because they predict digits as tokens rather than computing them.

Some design decisions:

- **Prices are supplied, not invented.** The assignment says to "assign" a price to each shoe. If the model invents prices, two runs (or two emails) can price the same shoe differently, which makes the output non-reproducible and impossible to check. A fixed price list in the prompt makes the math verifiable and keeps both emails consistent.
- **The model does the math, Python checks it.** The assignment asks the model to compute subtotals and the total, so the prompt asks for that — but a validation function recomputes everything independently. That way the notebook reports *whether* the model got the math right instead of assuming it did.
- **Temperature 0.** Extraction has one correct answer, so randomness only adds risk.
- **"JSON only" instruction plus a tolerant parser.** Open models often wrap JSON in ``` fences or add a sentence of preamble. Rather than fail, the parser pulls out the outermost `{...}` block.
- **The one-shot example is ground truth.** For the one-shot prompt, the example answer for Email 1 is built in Python from known-correct values, not copied from the zero-shot model output. If the zero-shot output had an error, reusing it as the example would teach the model that error.

In [2]:
EMAIL_1 = '''Hey Jim,

I'd like to order some new shoes. Please ship the following:

1. Nike Air Jordan I - 2 pair
2. Converse All-Star - 10 pair
3. New Balance 990 - 1 pair
4. Nike Zoom Fly 5 men's red - 2 pair

Please provide sub-totals and grand total cost.

Thanks.

Lance Gentry
123 Main St.
Chelsea, MI 48109
248-229-2229'''

EMAIL_2 = '''Hey Michelle,

I'd like to order some new shoes. Please ship the following:

1. Nike Air Jordan I - 1 pair
2. Converse All-Star - 20 pair
3. New Balance 990 - 2 pair
4. Nike Zoom Fly 5 men's red - 5 pair

Please provide sub-totals and grand total cost.

Thanks.

Artis Gilmore
723 Lexington Blvd.
New York, NY 10001
(503) 484-1029'''

# Assigned unit prices (USD) — used in the prompts AND for independent validation
PRICE_LIST = {
    "Nike Air Jordan I": 180.00,
    "Converse All-Star": 60.00,
    "New Balance 990": 200.00,
    "Nike Zoom Fly 5 men's red": 160.00,
}
price_text = "\n".join(f"- {k}: ${v:.2f} per pair" for k, v in PRICE_LIST.items())
print(price_text)

- Nike Air Jordan I: $180.00 per pair
- Converse All-Star: $60.00 per pair
- New Balance 990: $200.00 per pair
- Nike Zoom Fly 5 men's red: $160.00 per pair


### Helper functions

`call_llm` sends the messages, `extract_json` turns the raw reply into a Python dict, and `validate_order` recomputes the math and checks the extracted fields against what the email actually says.

In [3]:
def call_llm(client, model, messages, temperature=0.0, max_tokens=1200):
    resp = client.chat.completions.create(
        model=model, messages=messages, temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content

def extract_json(text):
    # Strip ``` fences / preamble and parse the outermost JSON object.
    cleaned = re.sub(r"```(?:json)?", "", text).strip()
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("No JSON object found in model output")
    return json.loads(cleaned[start:end + 1])

def validate_order(data, expected_qty):
    # Recompute every subtotal and the grand total; flag mismatches.
    problems, running = [], 0.0
    items = data.get("shoes", [])
    if len(items) != len(expected_qty):
        problems.append(f"Expected {len(expected_qty)} line items, got {len(items)}")
    for item in items:
        name = f"{item.get('shoe_brand','')} {item.get('shoe_model','')}".strip()
        qty, price, sub = item.get("shoe_quantity"), item.get("shoe_price"), item.get("shoe_subtotal")
        match = next((k for k in expected_qty if k.lower() in name.lower() or name.lower() in k.lower()), None)
        if match is None:
            problems.append(f"Unrecognized shoe: {name}")
            continue
        if qty != expected_qty[match]:
            problems.append(f"{match}: quantity {qty}, email says {expected_qty[match]}")
        if price is not None and abs(float(price) - PRICE_LIST[match]) > 0.01:
            problems.append(f"{match}: price {price}, price list says {PRICE_LIST[match]}")
        try:
            correct_sub = float(qty) * float(price)
            running += correct_sub
            if abs(float(sub) - correct_sub) > 0.01:
                problems.append(f"{match}: subtotal {sub}, should be {correct_sub:.2f}")
        except (TypeError, ValueError):
            problems.append(f"{match}: missing/non-numeric quantity, price, or subtotal")
    gt = data.get("grand_total")
    try:
        if abs(float(gt) - running) > 0.01:
            problems.append(f"grand_total {gt}, should be {running:.2f}")
    except (TypeError, ValueError):
        problems.append("grand_total missing or non-numeric")
    print("✅ All math and quantities check out" if not problems else "❌ Issues found:")
    for p in problems:
        print("   -", p)
    return problems

EMAIL_1_QTY = {"Nike Air Jordan I": 2, "Converse All-Star": 10, "New Balance 990": 1, "Nike Zoom Fly 5 men's red": 2}
EMAIL_2_QTY = {"Nike Air Jordan I": 1, "Converse All-Star": 20, "New Balance 990": 2, "Nike Zoom Fly 5 men's red": 5}

## Part A — Zero-shot prompt (Email 1)

Zero-shot means the model gets instructions but **no worked example**. Everything it knows about the desired output has to come from the description, so the prompt spells out every field name, the nesting (line items in a `shoes` array), the data types (numbers, not strings like `"$360"`), and the "JSON only" requirement.

The schema description is deliberately explicit about names like `shoe_agent_name` and `request_type`. Without that, a model asked to "extract all the information" will choose its own keys (`agent`, `name`, `order_items`...), and two runs may not even agree with each other.

In [4]:
ZERO_SHOT_SYSTEM = "You are a precise order-processing assistant. You output only valid JSON with no commentary."

zero_shot_prompt = f'''Extract all the information from the email below and return it as a single JSON object.

Use exactly these keys:
- shoe_agent_name: the person the email is addressed to
- request_type: the type of request (e.g. "Order")
- customer_name, customer_street, customer_city, customer_state, customer_zip, customer_phone
- shoes: an array with one object per line item, each containing:
    shoe_brand, shoe_model, shoe_quantity (integer),
    shoe_price (number, unit price from the price list),
    shoe_subtotal (number, shoe_quantity * shoe_price)
- grand_total: number, the sum of all shoe_subtotal values

Price list:
{price_text}

Return ONLY the JSON object. Numbers must be plain numbers (no $ signs, no quotes).

Email:
<email>
{EMAIL_1}
</email>'''

zs_messages = [
    {"role": "system", "content": ZERO_SHOT_SYSTEM},
    {"role": "user", "content": zero_shot_prompt},
]

zs_raw = call_llm(oa_client, OPENAI_MODEL, zs_messages)
print("RAW MODEL OUTPUT:\n", zs_raw)

RAW MODEL OUTPUT:
 {
  "shoe_agent_name": "Jim",
  "request_type": "Order",
  "customer_name": "Lance Gentry",
  "customer_street": "123 Main St.",
  "customer_city": "Chelsea",
  "customer_state": "MI",
  "customer_zip": "48109",
  "customer_phone": "248-229-2229",
  "shoes": [
    {
      "shoe_brand": "Nike",
      "shoe_model": "Air Jordan I",
      "shoe_quantity": 2,
      "shoe_price": 180,
      "shoe_subtotal": 360
    },
    {
      "shoe_brand": "Converse",
      "shoe_model": "All-Star",
      "shoe_quantity": 10,
      "shoe_price": 60,
      "shoe_subtotal": 600
    },
    {
      "shoe_brand": "New Balance",
      "shoe_model": "990",
      "shoe_quantity": 1,
      "shoe_price": 200,
      "shoe_subtotal": 200
    },
    {
      "shoe_brand": "Nike",
      "shoe_model": "Zoom Fly 5 men's red",
      "shoe_quantity": 2,
      "shoe_price": 160,
      "shoe_subtotal": 320
    }
  ],
  "grand_total": 1480
}


In [5]:
zs_json = extract_json(zs_raw)
print(json.dumps(zs_json, indent=2))
zs_problems = validate_order(zs_json, EMAIL_1_QTY)

{
  "shoe_agent_name": "Jim",
  "request_type": "Order",
  "customer_name": "Lance Gentry",
  "customer_street": "123 Main St.",
  "customer_city": "Chelsea",
  "customer_state": "MI",
  "customer_zip": "48109",
  "customer_phone": "248-229-2229",
  "shoes": [
    {
      "shoe_brand": "Nike",
      "shoe_model": "Air Jordan I",
      "shoe_quantity": 2,
      "shoe_price": 180,
      "shoe_subtotal": 360
    },
    {
      "shoe_brand": "Converse",
      "shoe_model": "All-Star",
      "shoe_quantity": 10,
      "shoe_price": 60,
      "shoe_subtotal": 600
    },
    {
      "shoe_brand": "New Balance",
      "shoe_model": "990",
      "shoe_quantity": 1,
      "shoe_price": 200,
      "shoe_subtotal": 200
    },
    {
      "shoe_brand": "Nike",
      "shoe_model": "Zoom Fly 5 men's red",
      "shoe_quantity": 2,
      "shoe_price": 160,
      "shoe_subtotal": 320
    }
  ],
  "grand_total": 1480
}
✅ All math and quantities check out


### ✏️ Your commentary — zero-shot result

*Replace these guiding questions with your own observations from your run:*

- Did the model return clean JSON, or did the parser have to strip fences/preamble?
- Did it use the exact key names requested, or rename/add any?
- How did it split brand vs. model? (e.g., is "Nike Zoom Fly 5 men's red" split as brand `Nike` / model `Zoom Fly 5 men's red`? Did it break color or gender into separate fields on its own?) Note that the prompt never defined where the brand ends and the model begins — this is a genuine ambiguity, not just a model error.
- How was the phone number formatted? Was the zip a string or a number?
- Did the validator find any math errors? If so, which kind — multiplication or the final sum?

## Part B — One-shot prompt (example = Email 1, target = Email 2)

One-shot prompting adds a single worked example. Here it is implemented as a **fake conversation turn**: a `user` message containing Email 1 followed by an `assistant` message containing the correct JSON, then a new `user` message containing Email 2. Chat models are trained on alternating turns, so showing the example *as a previous answer the assistant gave* tends to be followed more faithfully than pasting "Example:" text into one big prompt.

The instructions in this version are intentionally **shorter** than the zero-shot prompt. That's on purpose: the point of one-shot is that the example carries the format (key names, nesting, number types, brand/model split) so the instructions don't have to. Comparing Part A and Part B shows how much work the example does on its own.

The example JSON is built from code using the price list, so it's guaranteed correct.

In [6]:
def build_reference(agent, name, street, city, state, zip_code, phone, qty_map):
    shoes = []
    for full_name, qty in qty_map.items():
        brand, model = full_name.split(" ", 1)
        price = PRICE_LIST[full_name]
        shoes.append({
            "shoe_brand": brand, "shoe_model": model, "shoe_quantity": qty,
            "shoe_price": price, "shoe_subtotal": round(qty * price, 2),
        })
    return {
        "shoe_agent_name": agent, "request_type": "Order",
        "customer_name": name, "customer_street": street, "customer_city": city,
        "customer_state": state, "customer_zip": zip_code, "customer_phone": phone,
        "shoes": shoes,
        "grand_total": round(sum(s["shoe_subtotal"] for s in shoes), 2),
    }

example_answer = build_reference("Jim", "Lance Gentry", "123 Main St.", "Chelsea", "MI", "48109",
                                 "248-229-2229", EMAIL_1_QTY)
print(json.dumps(example_answer, indent=2))

{
  "shoe_agent_name": "Jim",
  "request_type": "Order",
  "customer_name": "Lance Gentry",
  "customer_street": "123 Main St.",
  "customer_city": "Chelsea",
  "customer_state": "MI",
  "customer_zip": "48109",
  "customer_phone": "248-229-2229",
  "shoes": [
    {
      "shoe_brand": "Nike",
      "shoe_model": "Air Jordan I",
      "shoe_quantity": 2,
      "shoe_price": 180.0,
      "shoe_subtotal": 360.0
    },
    {
      "shoe_brand": "Converse",
      "shoe_model": "All-Star",
      "shoe_quantity": 10,
      "shoe_price": 60.0,
      "shoe_subtotal": 600.0
    },
    {
      "shoe_brand": "New",
      "shoe_model": "Balance 990",
      "shoe_quantity": 1,
      "shoe_price": 200.0,
      "shoe_subtotal": 200.0
    },
    {
      "shoe_brand": "Nike",
      "shoe_model": "Zoom Fly 5 men's red",
      "shoe_quantity": 2,
      "shoe_price": 160.0,
      "shoe_subtotal": 320.0
    }
  ],
  "grand_total": 1480.0
}


In [7]:
ONE_SHOT_SYSTEM = f'''You are a precise order-processing assistant. Extract the order from the customer's email as JSON,
using the same structure as your previous answer. Price each shoe using this price list, compute each
shoe_subtotal as quantity * price, and grand_total as the sum of subtotals.

Price list:
{price_text}

Output only valid JSON.'''

os_messages = [
    {"role": "system", "content": ONE_SHOT_SYSTEM},
    {"role": "user", "content": EMAIL_1},
    {"role": "assistant", "content": json.dumps(example_answer, indent=2)},
    {"role": "user", "content": EMAIL_2},
]

os_raw = call_llm(oa_client, OPENAI_MODEL, os_messages)
print("RAW MODEL OUTPUT:\n", os_raw)

RAW MODEL OUTPUT:
 {
  "shoe_agent_name": "Michelle",
  "request_type": "Order",
  "customer_name": "Artis Gilmore",
  "customer_street": "723 Lexington Blvd.",
  "customer_city": "New York",
  "customer_state": "NY",
  "customer_zip": "10001",
  "customer_phone": "(503) 484-1029",
  "shoes": [
    {
      "shoe_brand": "Nike",
      "shoe_model": "Air Jordan I",
      "shoe_quantity": 1,
      "shoe_price": 180.0,
      "shoe_subtotal": 180.0
    },
    {
      "shoe_brand": "Converse",
      "shoe_model": "All-Star",
      "shoe_quantity": 20,
      "shoe_price": 60.0,
      "shoe_subtotal": 1200.0
    },
    {
      "shoe_brand": "New",
      "shoe_model": "Balance 990",
      "shoe_quantity": 2,
      "shoe_price": 200.0,
      "shoe_subtotal": 400.0
    },
    {
      "shoe_brand": "Nike",
      "shoe_model": "Zoom Fly 5 men's red",
      "shoe_quantity": 5,
      "shoe_price": 160.0,
      "shoe_subtotal": 800.0
    }
  ],
  "grand_total": 2580.0
}


In [8]:
os_json = extract_json(os_raw)
print(json.dumps(os_json, indent=2))
os_problems = validate_order(os_json, EMAIL_2_QTY)

{
  "shoe_agent_name": "Michelle",
  "request_type": "Order",
  "customer_name": "Artis Gilmore",
  "customer_street": "723 Lexington Blvd.",
  "customer_city": "New York",
  "customer_state": "NY",
  "customer_zip": "10001",
  "customer_phone": "(503) 484-1029",
  "shoes": [
    {
      "shoe_brand": "Nike",
      "shoe_model": "Air Jordan I",
      "shoe_quantity": 1,
      "shoe_price": 180.0,
      "shoe_subtotal": 180.0
    },
    {
      "shoe_brand": "Converse",
      "shoe_model": "All-Star",
      "shoe_quantity": 20,
      "shoe_price": 60.0,
      "shoe_subtotal": 1200.0
    },
    {
      "shoe_brand": "New",
      "shoe_model": "Balance 990",
      "shoe_quantity": 2,
      "shoe_price": 200.0,
      "shoe_subtotal": 400.0
    },
    {
      "shoe_brand": "Nike",
      "shoe_model": "Zoom Fly 5 men's red",
      "shoe_quantity": 5,
      "shoe_price": 160.0,
      "shoe_subtotal": 800.0
    }
  ],
  "grand_total": 2580.0
}
✅ All math and quantities check out


### Expected answer for Email 2

For reference, here is the correct output for Email 2 computed in Python. Comparing it field by field with the model's output shows exactly where (if anywhere) the model diverged.

In [9]:
expected_2 = build_reference("Michelle", "Artis Gilmore", "723 Lexington Blvd.", "New York", "NY", "10001",
                             "(503) 484-1029", EMAIL_2_QTY)
print(json.dumps(expected_2, indent=2))

print("\nField-by-field differences (top-level):")
for k in expected_2:
    if k != "shoes" and str(os_json.get(k)) != str(expected_2[k]):
        print(f"  {k}: model={os_json.get(k)!r}  expected={expected_2[k]!r}")

{
  "shoe_agent_name": "Michelle",
  "request_type": "Order",
  "customer_name": "Artis Gilmore",
  "customer_street": "723 Lexington Blvd.",
  "customer_city": "New York",
  "customer_state": "NY",
  "customer_zip": "10001",
  "customer_phone": "(503) 484-1029",
  "shoes": [
    {
      "shoe_brand": "Nike",
      "shoe_model": "Air Jordan I",
      "shoe_quantity": 1,
      "shoe_price": 180.0,
      "shoe_subtotal": 180.0
    },
    {
      "shoe_brand": "Converse",
      "shoe_model": "All-Star",
      "shoe_quantity": 20,
      "shoe_price": 60.0,
      "shoe_subtotal": 1200.0
    },
    {
      "shoe_brand": "New",
      "shoe_model": "Balance 990",
      "shoe_quantity": 2,
      "shoe_price": 200.0,
      "shoe_subtotal": 400.0
    },
    {
      "shoe_brand": "Nike",
      "shoe_model": "Zoom Fly 5 men's red",
      "shoe_quantity": 5,
      "shoe_price": 160.0,
      "shoe_subtotal": 800.0
    }
  ],
  "grand_total": 2580.0
}

Field-by-field differences (top-level):


## Part C — Controlled comparison: zero-shot on Email 2

Parts A and B differ in **two** ways at once — the prompting technique *and* the email. To isolate the effect of the example, this cell runs the exact zero-shot prompt from Part A on Email 2. Now zero-shot vs. one-shot can be compared on the same input.

In [10]:
zs2_messages = [
    {"role": "system", "content": ZERO_SHOT_SYSTEM},
    {"role": "user", "content": zero_shot_prompt.replace(EMAIL_1, EMAIL_2)},
]
zs2_raw = call_llm(oa_client, OPENAI_MODEL, zs2_messages)
zs2_json = extract_json(zs2_raw)
print(json.dumps(zs2_json, indent=2))
zs2_problems = validate_order(zs2_json, EMAIL_2_QTY)

{
  "shoe_agent_name": "Michelle",
  "request_type": "Order",
  "customer_name": "Artis Gilmore",
  "customer_street": "723 Lexington Blvd.",
  "customer_city": "New York",
  "customer_state": "NY",
  "customer_zip": "10001",
  "customer_phone": "(503) 484-1029",
  "shoes": [
    {
      "shoe_brand": "Nike",
      "shoe_model": "Air Jordan I",
      "shoe_quantity": 1,
      "shoe_price": 180,
      "shoe_subtotal": 180
    },
    {
      "shoe_brand": "Converse",
      "shoe_model": "All-Star",
      "shoe_quantity": 20,
      "shoe_price": 60,
      "shoe_subtotal": 1200
    },
    {
      "shoe_brand": "New Balance",
      "shoe_model": "990",
      "shoe_quantity": 2,
      "shoe_price": 200,
      "shoe_subtotal": 400
    },
    {
      "shoe_brand": "Nike",
      "shoe_model": "Zoom Fly 5 men's red",
      "shoe_quantity": 5,
      "shoe_price": 160,
      "shoe_subtotal": 800
    }
  ],
  "grand_total": 2580
}
✅ All math and quantities check out


In [11]:
# Side-by-side summary of the three Problem #1 runs
def summarize(label, data, problems):
    return {
        "run": label,
        "keys_match_spec": set(data.keys()) == set(example_answer.keys()),
        "line_items": len(data.get("shoes", [])),
        "first_item_brand/model": f"{data['shoes'][0].get('shoe_brand')} / {data['shoes'][0].get('shoe_model')}" if data.get("shoes") else None,
        "phone": data.get("customer_phone"),
        "zip_type": type(data.get("customer_zip")).__name__,
        "grand_total": data.get("grand_total"),
        "math_errors": len(problems),
    }

rows = [summarize("Zero-shot / Email 1", zs_json, zs_problems),
        summarize("One-shot / Email 2", os_json, os_problems),
        summarize("Zero-shot / Email 2", zs2_json, zs2_problems)]
for r in rows:
    print(json.dumps(r, indent=2))

{
  "run": "Zero-shot / Email 1",
  "keys_match_spec": true,
  "line_items": 4,
  "first_item_brand/model": "Nike / Air Jordan I",
  "phone": "248-229-2229",
  "zip_type": "str",
  "grand_total": 1480,
  "math_errors": 0
}
{
  "run": "One-shot / Email 2",
  "keys_match_spec": true,
  "line_items": 4,
  "first_item_brand/model": "Nike / Air Jordan I",
  "phone": "(503) 484-1029",
  "zip_type": "str",
  "grand_total": 2580.0,
  "math_errors": 0
}
{
  "run": "Zero-shot / Email 2",
  "keys_match_spec": true,
  "line_items": 4,
  "first_item_brand/model": "Nike / Air Jordan I",
  "phone": "(503) 484-1029",
  "zip_type": "str",
  "grand_total": 2580,
  "math_errors": 0
}


### ✏️ Your commentary — comparing the Problem #1 responses

*The assignment asks you to explicitly compare the similarities and differences between the responses. Use the summary table above and write this in your own words. Things worth addressing:*

**Similarities**
- Which fields did every run get right? (Names, addresses, and quantities are usually reliable because they're copied straight from the text.)
- Did all runs keep the same JSON shape?

**Differences**
- Did the one-shot output copy the example's formatting choices exactly (key order, `customer_zip` as a string, brand/model split, phone format)? Did the zero-shot outputs make different choices on those?
- Note the phone number: Email 2 uses `(503) 484-1029` while the example uses `248-229-2229`. Did the one-shot model keep the original format or "normalize" it to match the example? Either is defensible — what would a real order system want?
- Compare **one-shot / Email 2** with **zero-shot / Email 2** directly. Since the input is identical, any difference is caused by the example. Was the example worth the extra tokens?
- Math: Email 2 has larger numbers (20 × $60, 5 × $160). Did errors show up there that didn't in Email 1?

**Evaluation**
- Which technique would you trust in production, and would you still let the LLM do the arithmetic or have code compute totals after extraction? Why?
- If you switch to a different OpenAI model, does anything change?

---
# Problem #2 — Chain of Thought with OpenAI (re-creating Listing 6.4)

## Approach

Listing 6.4 is **few-shot chain-of-thought (CoT)** prompting: three solved word problems show the model *how to reason step by step* before stating an answer, then a fourth question is left open. The final question is a classic trap — the intuitive answer is 35 (half of 70), but the sister is always 3 years younger, so the correct answer is **67**.

Moving from Azure OpenAI to OpenAI only changes the client setup: Azure needs an endpoint, a deployment name, and an API version; OpenAI just needs an API key and a model name. The prompt itself carries over unchanged.

Three variants are run so the effect of CoT can actually be measured rather than assumed:

1. **Faithful re-creation** — the entire Q/A transcript in one user message, as in the listing.
2. **Multi-turn chat version** — each example Q as a `user` turn and each A as an `assistant` turn. Same content, but structured the way chat models are trained.
3. **Zero-shot baseline (no examples)** — only the last question, with a "answer with just a number" instruction to suppress the model's own reasoning. This shows what the examples are contributing.

One subtlety worth noting: the third example (Olivia's bagels) is **weaker** than the others — it skips the explicit arithmetic expression and never says "The answer is X." The examples are not perfectly consistent, and it's interesting to see which style the model imitates.

In [ ]:
COT_EXAMPLES = [
    ("There were nine computers in the server room. Five more computers were installed each day, from Monday to Thursday. How many computers are now in the server room?",
     "There are 4 days from Monday to Thursday. 5 computers were added each day. That means in total 4 * 5 = 20 computers were added. There were 9 computers initially, so now there are 9 + 20 = 29 computers. The answer is 29."),
    ("Michael had 58 golf balls. On Tuesday, he lost 23 golf balls. On Wednesday, he lost 2 more. How many golf balls did he have at the end of Wednesday?",
     "Michael initially had 58 balls. He lost 23 on Tuesday, so after that he has 58 - 23 = 35 balls. On Wednesday he lost 2 more so now he has 35 - 2 = 33 balls. The answer is 33."),
    ("Olivia has $23. She bought five bagels for $3 each. How much money does she have left?",
     "She bought 5 bagels for $3 each. This means she spent $15. She has $8 left."),
]
QUESTION = "When I was 6 my sister was half my age. Now I'm 70 how old is my sister?"

### Variant 1 — Faithful re-creation (single prompt)

In [ ]:
transcript = "\n".join(f"Q: {q}\nA: {a}" for q, a in COT_EXAMPLES) + f"\nQ: {QUESTION}\nA:"
print(transcript, "\n" + "=" * 60)

v1_messages = [
    {"role": "system", "content": "You are a helpful assistant that answers math word problems."},
    {"role": "user", "content": transcript},
]
v1 = call_llm(oa_client, OPENAI_MODEL, v1_messages, max_tokens=300)
print(v1)

### Variant 2 — Multi-turn chat structure

In [ ]:
v2_messages = [{"role": "system", "content": "You are a helpful assistant that answers math word problems."}]
for q, a in COT_EXAMPLES:
    v2_messages.append({"role": "user", "content": q})
    v2_messages.append({"role": "assistant", "content": a})
v2_messages.append({"role": "user", "content": QUESTION})

v2 = call_llm(oa_client, OPENAI_MODEL, v2_messages, max_tokens=300)
print(v2)

### Variant 3 — Baseline with no examples and no reasoning

In [ ]:
v3_messages = [
    {"role": "user", "content": QUESTION + " Respond with only the number, no explanation."},
]
v3 = call_llm(oa_client, OPENAI_MODEL, v3_messages, max_tokens=10)
print(v3)

In [ ]:
def final_number(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else None

for label, out in [("V1 single-prompt CoT", v1), ("V2 multi-turn CoT", v2), ("V3 no-CoT baseline", v3)]:
    ans = final_number(out)
    print(f"{label:24s} -> final number: {ans}  {'✅' if ans == '67' else '❌ (correct: 67)'}")

### Optional — consistency check

A single run at temperature 0 says little about reliability. Running each variant several times at a normal temperature shows whether a correct answer was robust or lucky. (Self-consistency — sampling several CoT answers and taking the majority — is itself a known prompting technique.)

In [ ]:
from collections import Counter

N = 5
for label, msgs, mt in [("V1", v1_messages, 300), ("V2", v2_messages, 300), ("V3", v3_messages, 10)]:
    answers = [final_number(call_llm(oa_client, OPENAI_MODEL, msgs, temperature=0.8, max_tokens=mt)) for _ in range(N)]
    print(f"{label}: {Counter(answers)}")

### ✏️ Your commentary — Problem #2

*Write this from your actual outputs. Questions to address:*

**Comparing the replies to each question / variant**
- Did V1 and V2 both reach 67? Did they phrase the reasoning the same way?
- Which example's *style* did the model copy — the explicit "X - Y = Z ... The answer is N." pattern from the first two, or the looser style of the Olivia example? Did it end with "The answer is ..."?
- Did the model's reasoning identify the key idea (the age *difference* of 3 years stays constant), or did it get the right number by some other path?
- What did the no-CoT baseline (V3) answer? If it said 35, that's the classic failure the listing is meant to demonstrate. If it said 67, what does that tell you about how much newer models have internalized this kind of puzzle compared with the models the textbook was written against?
- Did the consistency check reveal any variation that a single run hid?

**Evaluation**
- Did moving from Azure OpenAI to OpenAI change anything besides the client setup?
- Is single-prompt or multi-turn structure better here, and why might it matter more for smaller/older models?

---
## ✏️ Overall reflection

*Tie the two problems together in your own words: both are about getting a model to do math reliably. In Problem #1 the fix was structural (give the prices, verify in code); in Problem #2 it was making the model show its work. When would you use each approach, and when would you combine them?*